# 2. Ground filtering, terrain models, and volumes

From a raw cloud to bare-earth and surface rasters, object heights, and cut/fill volumes — all with no GPU and no training.

**Install** (raster export needs the GIS extra):

```bash
pip install "geoai3d[laz,gis]"
```

In [ ]:
from geoai3d import read_lidar, ground

cloud = read_lidar("../data/ahn_sample.laz")
classified = ground(cloud, cloth_resolution=1.0, class_threshold=0.5)
is_ground = classified.attribute("is_ground")
print(f"ground: {100 * is_ground.mean():.1f}% of points")

`ground` runs the Cloth Simulation Filter (a physical simulation, no labels) and adds a boolean `is_ground` column. Now rasterise the bare earth (DTM) and the top surface (DSM).

In [ ]:
from geoai3d import to_dtm, to_dsm

dtm = to_dtm(classified, resolution=0.5)
dsm = to_dsm(classified, resolution=0.5)
print("DTM grid:", dtm.shape, "resolution:", dtm.resolution)

## Object heights (normalised DSM)

Subtracting the terrain from the surface gives the height of everything above the ground — buildings and trees stand out.

In [ ]:
import numpy as np
from geoai3d import difference

ndsm = difference(dsm, dtm)
print("tallest object above ground:", round(float(np.nanmax(ndsm.data)), 1), "m")

## Volume above ground

`volume` integrates a height raster against a base level. On the normalised DSM, the *fill* volume is the built/vegetated volume standing above the bare earth.

In [ ]:
from geoai3d import volume

result = volume(ndsm, base=0.0)
print("above-ground volume (m^3):", round(result["fill"], 1))

## Preview the rasters

A quick heatmap of each raster (plotly, from the `viz` extra). The DTM is the smooth bare earth; the object-height raster lights up on buildings and trees.

In [ ]:
import plotly.express as px

px.imshow(dtm.data, color_continuous_scale="viridis",
          title="DTM — bare-earth elevation (m)")

In [ ]:
px.imshow(ndsm.data, color_continuous_scale="magma",
          title="Height above ground (m)")

## Export to GeoTIFF

The rasters carry their transform and CRS, so they open directly in QGIS or ArcGIS.

In [ ]:
from geoai3d import to_geotiff

to_geotiff(dtm, "dtm.tif")
to_geotiff(ndsm, "object_heights.tif")
print("wrote dtm.tif and object_heights.tif")

That is a complete terrain workflow — bare earth, surface, object heights, and volume — from one LiDAR tile.